In [1]:
import os
import pandas as pd
import random
import shutil
from ultralytics import YOLO
import cv2
import numpy as np
import torch


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/shuwuyou/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [23]:

# Load labels.csv with correct headers
labels_path = "labels.csv"
data_dir = "data/images"  # Adjust if needed

df = pd.read_csv(labels_path)
df = df[["image_file", "image_path", "xmin", "ymin", "xmax", "ymax"]]  # Select required columns

import os

notebook_dir = os.getcwd()  
dataset_dir = os.path.join(notebook_dir, "dataset")  

print(f"✅ Dataset directory set to: {dataset_dir}")

# Create YOLO-formatted dataset directories
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(dataset_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(dataset_dir, "labels", split), exist_ok=True)

# Split dataset (70% train, 15% val, 15% test)
random.seed(42)
df_shuffled = df.sample(frac=1).reset_index(drop=True)
train_split = int(0.7 * len(df))
val_split = int(0.85 * len(df))

train_df = df_shuffled[:train_split]
val_df = df_shuffled[train_split:val_split]
test_df = df_shuffled[val_split:]

# Convert bounding box to YOLO format
def convert_to_yolo(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[2]) / 2.0 * dw
    y = (box[1] + box[3]) / 2.0 * dh
    w = (box[2] - box[0]) * dw
    h = (box[3] - box[1]) * dh
    return f"0 {x} {y} {w} {h}"

# Process dataset
def process_data(df, split):
    for _, row in df.iterrows():
        img_filename = row["image_file"]
        label_filename = img_filename.replace(".jpeg", ".txt").replace(".jpg", ".txt").replace(".png", ".txt")

        img_src = os.path.join("data/images", img_filename)
        label_dst = os.path.join(dataset_dir, "labels", split, label_filename)

        # Check if image exists before copying
        if not os.path.exists(img_src):
            print(f"❌ Missing image: {img_src}, skipping label")
            continue

        # Read image dimensions to convert labels
        img = cv2.imread(img_src)
        if img is None:
            print(f"⚠️ Unable to read image: {img_src}")
            continue
        height, width, _ = img.shape

        # Convert bounding box to YOLO format
        yolo_label = convert_to_yolo((width, height), (row["xmin"], row["ymin"], row["xmax"], row["ymax"]))

        # Write label file
        with open(label_dst, "w") as f:
            f.write(yolo_label + "\n")

        # Debugging: Confirm label file is written
        if os.path.exists(label_dst):
            print(f"✅ Label saved: {label_dst}")
        else:
            print(f"❌ Failed to save label: {label_dst}")



# Process all dataset splits
process_data(train_df, "train")
process_data(val_df, "val")
process_data(test_df, "test")

# Create YOLOv8-compatible YAML configuration file
yaml_content = f"""
path: {dataset_dir}  # Root directory
train: images/train
val: images/val
test: images/test
nc: 1  # Number of classes (only 'car plate')
names: ['number_plate']
"""

yaml_path = os.path.join(dataset_dir, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)



✅ Dataset directory set to: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset
✅ Label saved: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train/N230.txt
✅ Label saved: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train/N21.txt
✅ Label saved: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train/N45.txt
✅ Label saved: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train/N97.txt
✅ Label saved: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train/N163.txt
✅ Label saved: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train/N172.txt
✅ Label saved: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train/N168.txt
✅ Label saved: /Users/shuwuyou/Deskt

In [24]:
# Train YOLOv8 model
model = YOLO("yolov8n.pt")  # Using YOLOv8 nano model for training
model.train(data=yaml_path, epochs=50, imgsz=640)

# Evaluate on test set
results = model.val()

# Inference on test images
test_images = os.listdir(os.path.join(dataset_dir, "images", "test"))
for img_name in test_images:
    img_path = os.path.join(dataset_dir, "images", "test", img_name)
    results = model(img_path)

    # Load ground truth
    gt_label_path = os.path.join(dataset_dir, "labels", "test", img_name.replace(".jpeg", ".txt"))
    with open(gt_label_path, "r") as f:
        gt_box = f.readline().strip().split()[1:]  # Ignore class ID
    
    gt_box = [float(x) for x in gt_box]

    # Draw results
    img = cv2.imread(img_path)
    h, w, _ = img.shape

    # Convert back YOLO format to pixel coordinates
    gt_x = int(gt_box[0] * w)
    gt_y = int(gt_box[1] * h)
    gt_w = int(gt_box[2] * w)
    gt_h = int(gt_box[3] * h)

    cv2.rectangle(img, (gt_x - gt_w // 2, gt_y - gt_h // 2), (gt_x + gt_w // 2, gt_y + gt_h // 2), (0, 255, 0), 2)

    # Draw detected boxes
    for res in results:
        for box in res.boxes.xywh:
            x, y, w, h = map(int, box)
            cv2.rectangle(img, (x - w // 2, y - h // 2), (x + w // 2, y + h // 2), (255, 0, 0), 2)

    # Save output image
    output_path = img_path.replace(".jpeg", "_output.jpeg")
    cv2.imwrite(output_path, img)

print("Training complete. Results saved in:", dataset_dir)


New https://pypi.org/project/ultralytics/8.3.82 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.78 🚀 Python-3.9.21 torch-2.6.0 CPU (Apple M1 Pro)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/data.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train14, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, re

train: Scanning /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train... 157 images, 0 backgrounds, 0 corrupt: 100%|██████████| 157/157 [00:00<00:00, 1905.99it/s]

train: New cache created: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/train.cache



val: Scanning /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/val... 34 images, 0 backgrounds, 0 corrupt: 100%|██████████| 34/34 [00:00<00:00, 2033.14it/s]

val: New cache created: /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/val.cache
Plotting labels to /opt/homebrew/runs/detect/train14/labels.jpg... 


...


50 epochs completed in 0.980 hours.
Optimizer stripped from /opt/homebrew/runs/detect/train14/weights/last.pt, 6.2MB
Optimizer stripped from /opt/homebrew/runs/detect/train14/weights/best.pt, 6.2MB

Validating /opt/homebrew/runs/detect/train14/weights/best.pt...
Ultralytics 8.3.78 🚀 Python-3.9.21 torch-2.6.0 CPU (Apple M1 Pro)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:05<00:00,  2.69s/it]


                   all         34         34      0.994      0.971      0.993      0.721
Speed: 1.4ms preprocess, 151.2ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /opt/homebrew/runs/detect/train14
MLflow: results logged to /opt/homebrew/runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'
Ultralytics 8.3.78 🚀 Python-3.9.21 torch-2.6.0 CPU (Apple M1 Pro)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/labels/val.cache... 34 images, 0 backgrounds, 0 corrupt: 100%|██████████| 34/34 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:05<00:00,  1.87s/it]


                   all         34         34      0.994      0.971      0.993      0.725
Speed: 0.5ms preprocess, 157.7ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /opt/homebrew/runs/detect/train142

image 1/1 /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/images/test/N123.jpeg: 640x640 5 number_plates, 105.9ms
Speed: 2.3ms preprocess, 105.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/images/test/N174.jpeg: 640x640 1 number_plate, 76.3ms
Speed: 2.7ms preprocess, 76.3ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/images/test/N158.jpeg: 640x480 1 number_plate, 62.9ms
Speed: 1.4ms preprocess, 62.9ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /Users/shuwuyou/Desk

In [ ]:
import cv2
import os
from ultralytics import YOLO

# Load trained model
model_path = "/opt/homebrew/runs/detect/train14/weights/best.pt"  # Adjust if needed
model = YOLO(model_path)

# Define test image directory
test_images_dir = "dataset/images/test"
output_dir = "dataset/images/test_output"
os.makedirs(output_dir, exist_ok=True)

# Loop through test images
for img_name in os.listdir(test_images_dir):
    img_path = os.path.join(test_images_dir, img_name)
    results = model(img_path)

    # Load image
    img = cv2.imread(img_path)

    # Draw detected bounding boxes
    for res in results:
        for box in res.boxes.xyxy:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Save output image
    output_path = os.path.join(output_dir, img_name)
    cv2.imwrite(output_path, img)

print(f"✅ Processed test images saved in {output_dir}")




image 1/1 /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/images/test/N123.jpeg: 640x640 5 number_plates, 96.0ms
Speed: 2.6ms preprocess, 96.0ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/images/test/N174.jpeg: 640x640 1 number_plate, 120.6ms
Speed: 3.5ms preprocess, 120.6ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/images/test/N52_output.jpeg: 544x640 1 number_plate, 76.5ms
Speed: 1.7ms preprocess, 76.5ms inference, 0.4ms postprocess per image at shape (1, 3, 544, 640)

image 1/1 /Users/shuwuyou/Desktop/NU/Winter/deep_learning/DMFP/Car-Plate-detection-OCR/dataset/images/test/N158.jpeg: 640x480 1 number_plate, 65.5ms
Speed: 1.4ms preprocess, 65.5ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 480)

im